# Librerías

In [269]:
# Librerías
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Módulos 
from astroquery.mpc import MPC
from math import ceil

# Cobs API

In [270]:
# Verificar la conexión a internet.
def verificar_conexion():
    try:
        requests.get("http://www.google.com", timeout=5)
        print('✅ Conectado a internet.')
        return True
    
    except requests.ConnectionError:
        print('🛑 Sin conexión a internet.')
        return False

In [271]:
# Conexión con la API de COBS
try:
    content = [] 
    fecha_inicial = '2000-01-01'
    nombre_cometa = 'C/2023 A3'
    Link_cops_API_pagina_1 = f'https://cobs.si/api/obs_list.api?des={nombre_cometa}&format=json&from_date={fecha_inicial}&page=1&exclude_faint=False&exclude_not_accurate=False'

    if verificar_conexion():
        print(f'⌛ Conectando con la base de datos [COBS Observaciones].')
        response_pagina_1 = requests.get(Link_cops_API_pagina_1)

        if response_pagina_1.status_code == 200:
            content_pagina_1 = response_pagina_1.json()
            numero_de_paginas = int(content_pagina_1['info']['pages'])

            content.extend(content_pagina_1['objects'])

            for pagina in range(2, numero_de_paginas + 1):
                Link_cops_API_pagina = f'https://cobs.si/api/obs_list.api?des={nombre_cometa}&format=json&from_date={fecha_inicial}&page={pagina}&exclude_faint=False&exclude_not_accurate=False'
                response_pagina = requests.get(Link_cops_API_pagina)
                content_pagina = response_pagina.json()
                content.extend(content_pagina['objects'])

            print('✅ Base de datos actualizada [COBS Observaciones].')
        
except requests.ConnectionError:
    print(f'🛑 Se presentó un error al cargar la base de datos.\nError: {response_pagina.status_code}\n{response_pagina.content}')

✅ Conectado a internet.
⌛ Conectando con la base de datos [COBS Observaciones].
✅ Base de datos actualizada [COBS Observaciones].


In [272]:
# Creación del data frame Cometa
cometa_df = pd.DataFrame(content)
cometa_df.__len__()

2975

In [273]:
# Numero de registros y variables sin filtrar la información
filas,columnas = cometa_df.shape
print(f'Registros: {filas}\nVariables: {columnas}')

Registros: 2975
Variables: 47


In [274]:
# Base de datos arrojada por la API
# cometa_df.sample(5)

In [275]:
# Métodos de observación
# cometa_df.obs_method.apply(
#     lambda registro: f"{registro['key']}: {registro['name']}" if (registro is not None) and ('key' in registro) and ('name' in registro) else 'Datos faltantes'
# ).value_counts()

In [276]:
# Tratamiento de los datos de interés
# cometa_df['obs_method_key'] = cometa_df.obs_method.apply(lambda registro: registro['key'] if registro is not None and 'key' in registro else 'Dato faltante')
cometa_df['obs_date'] = pd.to_datetime(pd.to_datetime(cometa_df.obs_date).dt.date)
cometa_df['magnitude'] = pd.to_numeric(cometa_df.magnitude)

In [277]:
# Creación del data frame curva de luz cruda
curva_de_luz_cruda_df = cometa_df[
        [
        # 'obs_method_key', 
        'obs_date', 
        'magnitude'
        ]
    ].copy()

# curva_de_luz_cruda_df.info()

In [278]:
# Numero de registros y variables con la información filtrada
filas,columnas = curva_de_luz_cruda_df.shape
print(f'Registros: {filas}\nVariables: {columnas}')

Registros: 2975
Variables: 2


In [279]:
# Data Frame de la curva de luz
# curva_de_luz_cruda_df.sample(5)

In [280]:
# Curva de luz cruda.
labels = {
    'obs_date':'Observation Date',
    'magnitude':'Apparent total magnitude', 
    # 'obs_method_key' : 'Observation Method'
    }

fig = px.scatter(curva_de_luz_cruda_df, x='obs_date', y='magnitude', template= 'plotly_dark', labels= labels, title= f'Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")

fig.show()

# Perihelio Cobs API

In [324]:
# Conexión con la API de COBS para obtener el perihelio
try: 
    Link_cops_API = f'https://cobs.si/api/comet.api?des={nombre_cometa}'

    if verificar_conexion():
        print(f'⌛ Conectando con la base de datos [COBS Comets].')

        response = requests.get(Link_cops_API)

        if response.status_code == 200:
            perihelio = pd.to_datetime(response.json()['object']['perihelion_date'])
            print('✅ Perihelio del cometa obtenido.')
    
except requests.ConnectionError:
    print(f'🛑 Se presentó un error al cargar la base de datos.\nError: {response.status_code}\n{response.content}')

✅ Conectado a internet.
⌛ Conectando con la base de datos [COBS Comets].
✅ Perihelio del cometa obtenido.


# MPC API usando astroquery.

In [282]:
# Creación de data frame Ephemeris (conexión con la API del MPC)
efemerides_total = []

fecha_inicial = curva_de_luz_cruda_df.obs_date.min()
fecha_final = curva_de_luz_cruda_df.obs_date.max()
fechas = (fecha_final - fecha_inicial).days + 1

print('⌛ Conectando con la base de datos [MPC efemerides].')
for i in range(ceil(fechas/1441)):
    efemerides = MPC.get_ephemeris(nombre_cometa, start = str(fecha_inicial), number = 1441)  # type: ignore
    efemerides_ciclo_df = efemerides.to_pandas()
    efemerides_total.append(efemerides_ciclo_df)

    fecha_inicial = efemerides_ciclo_df.Date.max()

# Creación del data frame efemerides filtrada
efemerides_df = pd.concat(efemerides_total)
efemerides_df.columns = efemerides_df.columns.str.lower().str.replace(' ', '_')

efemerides_filtrada_df = efemerides_df[['date', 'delta','r', 'phase']].copy()
efemerides_filtrada_df = efemerides_filtrada_df.rename(columns = {'date':'obs_date'})
efemerides_filtrada_df['obs_date'] = pd.to_datetime(pd.to_datetime(efemerides_filtrada_df.obs_date).dt.date)
efemerides_filtrada_df.reset_index(inplace = True)

print('✅ Base de datos actualizada [MPC efemerides].')

# efemerides_filtrada_df

⌛ Conectando con la base de datos [MPC efemerides].
✅ Base de datos actualizada [MPC efemerides].


In [283]:
# Info del data frame ephemeris
# efemerides_df.info()

In [284]:
# Dar a los datos el formato deseado
efemerides_df.date = pd.to_datetime(efemerides_df.date)
efemerides_df.date = pd.to_datetime(efemerides_df.date.dt.date)
# efemerides_df.dtypes

In [285]:
# Creación del data frame ephemeris filtrada
efemerides_filtrada_df = efemerides_df[['date', 'delta','r', 'phase']].copy()
efemerides_filtrada_df = efemerides_filtrada_df.rename(columns = {'date':'obs_date'})
# efemerides_filtrada_df

# Unión de las bases de datos.

In [286]:
# Unión de las bases de datos COBS y MPC
curva_de_luz_procesada_df = curva_de_luz_cruda_df.merge(efemerides_filtrada_df, on='obs_date')
# curva_de_luz_procesada_df

In [287]:
# Información del data frame curva de lus procesada
# curva_de_luz_procesada_df.info()

In [288]:
# Reducción de la magnitud aparente y calculo del Delta t
beta = 0

curva_de_luz_procesada_df['magnitud_reducida'] = (
    curva_de_luz_cruda_df['magnitude'] 
    - 5 * np.log10(curva_de_luz_procesada_df['delta'] * curva_de_luz_procesada_df['r'])
    - (beta * curva_de_luz_procesada_df['phase'])
    )

curva_de_luz_procesada_df['delta_t'] = (curva_de_luz_procesada_df.obs_date - perihelio) # type: ignore
curva_de_luz_procesada_df['delta_t'] = curva_de_luz_procesada_df.delta_t.apply(lambda delta_t: delta_t.days)

# curva_de_luz_procesada_df

In [289]:
# Curva de luz reducida
# labels = {'obs_date':'Observation Date','magnitud_reducida':'Apparent total magnitude processed', 'obs_method_key' : 'Observation Method'}
# fig = px.scatter(curva_de_luz_procesada_df, x='obs_date', y='magnitud_reducida', color='obs_method_key', template= 'plotly_dark', labels= labels, title=f'Reduced Lightcurve of comet {nombre_cometa}')
# fig.update_yaxes(autorange="reversed")
# fig.show()

In [290]:
# Curva de luz reducida
labels = {
    'delta_t':'t-Δt',
    'magnitud_reducida':'Apparent total magnitude processed', 
    # 'obs_method_key' : 'Observation Method'
    }

fig = px.scatter(curva_de_luz_procesada_df, x='delta_t', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Reduced Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [291]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 9

curva_de_luz_promediada_df = curva_de_luz_procesada_df.copy()
curva_de_luz_promediada_df['promedio_movil'] = curva_de_luz_promediada_df.magnitud_reducida.rolling(window = numero_elementos_grupo).mean()
# curva_de_luz_promediada_df

In [292]:
# # Curva de luz Promediada
# labels = {
#     'obs_date':'Observation Date',
#     'magnitud_reducida':'Max apparent total magnitude reduced',
#     'obs_method_key' : 'Observation Method'
#     }

# fig = px.scatter(curva_de_luz_promediada_df, x='obs_date', y='promedio_movil', color='obs_method_key', template= 'plotly_dark', labels= labels, title= f'Average Lightcurve of comet {nombre_cometa}')
# fig.update_yaxes(autorange="reversed")
# fig.show()

In [293]:
# Curva de luz Promediada
labels = {
    'delta_t':'t - Δt',
    'magnitud_reducida':'Max apparent total magnitude reduced',
    'obs_method_key' : 'Observation Method'
    }

fig = px.scatter(curva_de_luz_promediada_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title= f'Average Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz interna (Envolvente inferior) v1 (Promedio corrido -> agrupación)

In [294]:
# Creación del data frame curva de luz agrupada
curva_de_luz_interna_v1_df = curva_de_luz_promediada_df.groupby(by = 'obs_date').max()
curva_de_luz_interna_v1_df = curva_de_luz_interna_v1_df.reset_index()

# curva_de_luz_interna_v1_df

In [295]:
# Gráfica de luz interna
# labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
# fig = px.scatter(curva_de_luz_interna_v1_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
# fig.update_traces(marker=dict(color='red', size=6, line=dict(width=1, color='DarkSlateGrey')))
# fig.update_yaxes(autorange="reversed")
# fig.show()

In [296]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v1_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz Externa (Envolvente superior) v1 (Promedio corrido -> agrupación)

In [297]:
# Creación del data frame curva de luz agrupada
curva_de_luz_externa_v1_df = curva_de_luz_promediada_df.groupby(by = 'obs_date').min()
curva_de_luz_externa_v1_df = curva_de_luz_externa_v1_df.reset_index()
# curva_de_luz_externa_v1_df

In [298]:
# Gráfica de lus promediada
# labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
# fig = px.scatter(curva_de_luz_externa_v1_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max averaged Lightcurve of comet {nombre_cometa}')
# fig.update_traces(marker=dict(color='yellow', size=6, line= dict(width=1, color='DarkSlateGrey')))
# fig.update_yaxes(autorange="reversed")
# fig.show()

In [299]:
# Gráfica de lus promediada
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v1_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=6, line= dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz interna (Envolvente inferior) v2 (Agrupación  -> promedio corrido)

In [300]:
# curva_de_luz_procesada_df.info()

In [301]:
# Creación del data frame curva de luz agrupada
curva_de_luz_agrupada_max_v2_df = curva_de_luz_procesada_df.groupby(by = 'obs_date').max()
curva_de_luz_agrupada_max_v2_df = curva_de_luz_agrupada_max_v2_df.reset_index()
# curva_de_luz_agrupada_max_v2_df

In [302]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_agrupada_max_v2_df, x='delta_t', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Min Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [303]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 7

curva_de_luz_interna_v2_df = curva_de_luz_agrupada_max_v2_df.copy()
curva_de_luz_interna_v2_df['promedio_movil'] = curva_de_luz_interna_v2_df.magnitud_reducida.rolling(window = numero_elementos_grupo, center= True).mean()
# curva_de_luz_interna_v2_df

In [304]:
# Gráfica de luz interna
# labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
# fig = px.scatter(curva_de_luz_interna_v2_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
# fig.update_traces(marker=dict(color='red', size=5, line=dict(width=1, color='DarkSlateGrey')))
# fig.update_yaxes(autorange="reversed")
# fig.show()

In [305]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_interna_v2_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='red', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz Externa (Envolvente superior) v2 (Agrupación  -> promedio corrido)

In [306]:
# Creación del data frame curva de luz agrupada
curva_de_luz_agrupada_min_v2_df = curva_de_luz_procesada_df.groupby(by = 'obs_date').min()
curva_de_luz_agrupada_min_v2_df = curva_de_luz_agrupada_min_v2_df.reset_index()
# curva_de_luz_agrupada_min_v2_df

In [307]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_agrupada_min_v2_df, x='delta_t', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Min Averaged Lightcurve of comet {nombre_cometa}')
fig.update_yaxes(autorange="reversed")
fig.show()

In [308]:
# Creación del data frame curva de luz promediada
numero_elementos_grupo = 7

curva_de_luz_externa_v2_df = curva_de_luz_agrupada_min_v2_df.copy()
curva_de_luz_externa_v2_df['promedio_movil'] = curva_de_luz_externa_v2_df.magnitud_reducida.rolling(window = numero_elementos_grupo, center= True).mean()
# curva_de_luz_externa_v2_df

In [309]:
# # Gráfica de luz interna
# labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
# fig = px.scatter(curva_de_luz_externa_v2_df, x='obs_date', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max Averaged Lightcurve of comet {nombre_cometa}')
# fig.update_traces(marker=dict(color='yellow', size=5, line=dict(width=1, color='DarkSlateGrey')))
# fig.update_yaxes(autorange="reversed")
# fig.show()

In [310]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_externa_v2_df, x='delta_t', y='promedio_movil', template= 'plotly_dark', labels= labels, title=f'Max Averaged Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='yellow', size=5, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz usando la mediada v1 (Mediana de cada día registrado).

In [311]:
# Creación del data frame curva de luz mediana v1
curva_de_luz_mediada_v1_df = curva_de_luz_procesada_df.groupby(by= 'obs_date').median(numeric_only= True)
curva_de_luz_mediada_v1_df = curva_de_luz_mediada_v1_df.reset_index()
# curva_de_luz_mediada_v1_df

In [312]:
# Gráfica de luz interna
# labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
# fig = px.scatter(curva_de_luz_mediada_v1_df, x='obs_date', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
# fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
# fig.update_yaxes(autorange="reversed")
# fig.show()

In [313]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v1_df, x='delta_t', y='magnitud_reducida', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Curva de luz usando la mediada v2 (Mediana de las dos curvas).

In [314]:
# Creación del data frame curva de luz mediana v2
curva_de_luz_mediada_v2_df = curva_de_luz_externa_v2_df.copy()
curva_de_luz_mediada_v2_df['mediana'] = (curva_de_luz_interna_v2_df['promedio_movil'] + curva_de_luz_externa_v2_df['promedio_movil'])/2
# curva_de_luz_mediada_v2_df

In [315]:
# Gráfica de luz interna
# labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
# fig = px.scatter(curva_de_luz_mediada_v2_df, x='obs_date', y='mediana', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
# fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
# fig.update_yaxes(autorange="reversed")
# fig.show()

In [316]:
# Gráfica de luz interna
labels = {'delta_t':'t - Δt','magnitud_reducida':'Magnitude reduced'}
fig = px.scatter(curva_de_luz_mediada_v2_df, x='delta_t', y='mediana', template= 'plotly_dark', labels= labels, title=f'Mediated Lightcurve of comet {nombre_cometa}')
fig.update_traces(marker=dict(color='aquamarine', size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.update_yaxes(autorange="reversed")
fig.show()

# Comparación de las curvas de luz v1 (Promedio corrido -> agrupación)

In [317]:
# Gráfica de luz promediada
# labels = {'obs_date':'Observation Date','magnitud_reducida':'Magnitude reduced'}
# fig = go.Figure()
# fig.add_trace(go.Scatter(x=curva_de_luz_externa_v1_df.obs_date, y=curva_de_luz_externa_v1_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.obs_date, y=curva_de_luz_interna_v1_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.obs_date, y=curva_de_luz_mediada_v1_df.magnitud_reducida, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
# # fig.add_trace(go.Scatter(x=curva_de_luz_externa_v2_df.obs_date, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente_v2', marker=dict(color='green', line=dict(width=1, color='DarkSlateGrey'))))
# # fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.obs_date, y=curva_de_luz_interna_v2_df.promedio_movil, mode='markers', name='Núcleo_v2', marker=dict(color='blue', line=dict(width=1, color='DarkSlateGrey'))))
# # fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.obs_date, y=curva_de_luz_mediada_v2_df.mediana, mode='markers', name='Mediana_v2', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
# fig.update_layout(template='plotly_dark')
# fig.update_yaxes(autorange="reversed")
# fig.update_layout(template='plotly_dark', xaxis_title='Observation Date', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')
# fig.show()

In [318]:
# Gráfica de luz promediada
fig = go.Figure()
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v1_df.delta_t, y=curva_de_luz_externa_v1_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v1_df.delta_t, y=curva_de_luz_interna_v1_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_mediada_v1_df.delta_t, y=curva_de_luz_mediada_v1_df.magnitud_reducida, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))
fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='t - Δt', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')
fig.show()

# Comparación de las curvas de luz v2 (Agrupación -> promedio corrido)

In [319]:
# Gráfica de luz promediada
fig = go.Figure()

fig.add_trace(go.Scatter(x=curva_de_luz_agrupada_min_v2_df.delta_t, y=curva_de_luz_agrupada_min_v2_df.magnitud_reducida, mode='markers', name='máximo diario', marker=dict(color="#fa00e9", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_agrupada_max_v2_df.delta_t, y=curva_de_luz_agrupada_max_v2_df.magnitud_reducida, mode='markers', name='minimo diario', marker=dict(color="#02FA61", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_externa_v2_df.delta_t, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente', marker=dict(color='yellow', line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.delta_t, y=curva_de_luz_interna_v2_df.promedio_movil, mode='markers', name='Núcleo', marker=dict(color='red', line=dict(width=1, color='DarkSlateGrey'))))
# fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.delta_t, y=curva_de_luz_mediada_v2_df.mediana, mode='markers', name='Mediana', marker=dict(color='aquamarine', line=dict(width=1, color='DarkSlateGrey'))))

fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='t - Δt', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')

fig.show()

# Envolvente superior calculada con percentiles v3

In [320]:
# Creación del data frame curva de luz agrupada
curva_de_luz_agrupada_min_v3_df = curva_de_luz_procesada_df.groupby(by = 'obs_date').min()
curva_de_luz_agrupada_min_v3_df = curva_de_luz_agrupada_min_v3_df.reset_index()
# curva_de_luz_agrupada_min_v3_df

In [321]:
curva_de_luz_interna_v3_df = curva_de_luz_agrupada_min_v3_df.copy()
curva_de_luz_interna_v3_df['percentil_movil'] = curva_de_luz_agrupada_min_v3_df.magnitud_reducida.rolling(window=3, center=True).apply(
    lambda x: np.percentile(x, 0), raw=True)

# curva_de_luz_externa_v2_df

In [322]:
# Gráfica de luz promediada
fig = go.Figure()

fig.add_trace(go.Scatter(x=curva_de_luz_procesada_df.delta_t, y=curva_de_luz_procesada_df.magnitud_reducida, mode='markers', name='magnitud reducida', marker=dict(color= '#00d9ff',line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_agrupada_min_v3_df.delta_t, y=curva_de_luz_agrupada_min_v3_df.magnitud_reducida, mode='markers', name='máximo diario', marker=dict(color="#fa00d4", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v3_df.delta_t, y=curva_de_luz_interna_v3_df.percentil_movil, mode='markers', name='Envolvente Percentil', marker=dict(color="#fe8402", line=dict(width=1, color='DarkSlateGrey'))))
fig.add_trace(go.Scatter(x=curva_de_luz_interna_v2_df.delta_t, y=curva_de_luz_externa_v2_df.promedio_movil, mode='markers', name='Envolvente Promedio', marker=dict(color="#FFEE00", line=dict(width=1, color='DarkSlateGrey'))))

fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='t - Δt', yaxis_title='Averaged Magnitude', title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')

fig.show()

# Guardar datos

In [323]:
curva_de_luz_procesada_df.to_csv(r'Bases_de_datos/curva_de_luz_procesada_COBS.txt', index=False)
curva_de_luz_interna_v2_df.to_csv(r'Bases_de_datos/curva_de_luz_interna_COBS.txt', index=False)
curva_de_luz_externa_v2_df.to_csv(r'Bases_de_datos/curva_de_luz_externa_COBS.txt', index=False)